# ML Basics For Energy Workflows

This notebook supports Session 1 of the training.

## Objectives
- Load synthetic production forecasting data
- Build a simple next-month oil-rate model
- Evaluate prediction quality
- Visualize forecast behavior for a few wells
- Connect the output back to an engineering decision workflow

## Suggested discussion
As you run this notebook, keep asking:
- What is the target?
- Which features are actually useful?
- What would an engineer do with this output?
- Where does GenAI help around this workflow?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")

candidate_paths = [
    Path("../../data/production_forecasting/well_monthly_production.csv"),
    Path("training/data/production_forecasting/well_monthly_production.csv"),
    Path("data/production_forecasting/well_monthly_production.csv"),
]

csv_path = next((path for path in candidate_paths if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate well_monthly_production.csv from this notebook.")

df = pd.read_csv(csv_path, parse_dates=["date"])
df = df.sort_values(["well_id", "date"]).reset_index(drop=True)

df.head()

In [ ]:
model_df = df.copy()
model_df["oil_rate_lag1"] = model_df.groupby("well_id")["oil_rate_bopd"].shift(1)
model_df["water_cut_lag1"] = model_df.groupby("well_id")["water_cut_pct"].shift(1)
model_df["uptime_lag1"] = model_df.groupby("well_id")["uptime_pct"].shift(1)
model_df = model_df.dropna(subset=["forecast_target_next_month_bopd"]).copy()
model_df["forecast_target_next_month_bopd"] = model_df["forecast_target_next_month_bopd"].astype(float)

feature_columns = [
    "oil_rate_bopd",
    "gas_rate_mscfd",
    "water_cut_pct",
    "uptime_pct",
    "choke_size_64in",
    "tubing_pressure_psi",
    "casing_pressure_psi",
    "gor_scf_bbl",
    "bhp_proxy_psi",
    "well_age_months",
    "lateral_length_ft",
    "completion_stage_count",
    "event_flag",
    "oil_rate_lag1",
    "water_cut_lag1",
    "uptime_lag1",
    "pad",
    "artificial_lift_type",
    "reservoir_zone",
]

target_column = "forecast_target_next_month_bopd"
train_df = model_df[model_df["date"] < model_df["date"].max() - pd.DateOffset(months=6)].copy()
test_df = model_df[model_df["date"] >= model_df["date"].max() - pd.DateOffset(months=6)].copy()

numeric_features = [col for col in feature_columns if col not in ["pad", "artificial_lift_type", "reservoir_zone"]]
categorical_features = ["pad", "artificial_lift_type", "reservoir_zone"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=250, random_state=42, max_depth=8)),
])

model.fit(train_df[feature_columns], train_df[target_column])
test_df["prediction_bopd"] = model.predict(test_df[feature_columns])

mae = mean_absolute_error(test_df[target_column], test_df["prediction_bopd"])
rmse = mean_squared_error(test_df[target_column], test_df["prediction_bopd"], squared=False)
r2 = r2_score(test_df[target_column], test_df["prediction_bopd"])

print(f"Rows used for modeling: {len(model_df)}")
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"MAE:  {mae:,.1f} bopd")
print(f"RMSE: {rmse:,.1f} bopd")
print(f"R^2:  {r2:0.3f}")

test_df[["well_id", "date", target_column, "prediction_bopd"]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=test_df,
    x=target_column,
    y="prediction_bopd",
    hue="pad",
    ax=axes[0],
)
axes[0].plot([test_df[target_column].min(), test_df[target_column].max()], [test_df[target_column].min(), test_df[target_column].max()], color="black", linestyle="--")
axes[0].set_title("Actual vs Predicted Next-Month Oil Rate")
axes[0].set_xlabel("Actual bopd")
axes[0].set_ylabel("Predicted bopd")

sample_wells = test_df.groupby("well_id")["prediction_bopd"].mean().sort_values(ascending=False).head(3).index.tolist()
plot_df = pd.concat([
    df[df["well_id"].isin(sample_wells)][["well_id", "date", "oil_rate_bopd"]].rename(columns={"oil_rate_bopd": "value"}).assign(series="historical oil rate"),
    test_df[test_df["well_id"].isin(sample_wells)][["well_id", "date", "prediction_bopd"]].rename(columns={"prediction_bopd": "value"}).assign(series="predicted next month"),
])

sns.lineplot(data=plot_df, x="date", y="value", hue="well_id", style="series", ax=axes[1])
axes[1].set_title("Sample Well History And Forecast Signal")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("bopd")
plt.tight_layout()
plt.show()

summary = test_df.copy()
summary["forecast_error_bopd"] = summary["prediction_bopd"] - summary[target_column]
summary["abs_error_bopd"] = summary["forecast_error_bopd"].abs()
summary.sort_values("abs_error_bopd", ascending=False)[[
    "well_id",
    "date",
    target_column,
    "prediction_bopd",
    "forecast_error_bopd",
    "water_cut_pct",
    "uptime_pct",
    "event_type",
]].head(10)

## Reflection Questions

Use the outputs above to discuss:

1. Which wells would you prioritize for surveillance, and why?
2. Which variables seem to be associated with underperformance?
3. What important engineering context is still missing from the model?
4. If an LLM helped write and explain this workflow, what parts would still need human review?

## Extension Ideas
- Compare this model against a simple decline-curve-style baseline.
- Add rolling averages or longer lag features.
- Create a management-ready summary table for the worst-performing wells.
- Ask an LLM to turn this notebook into a small `Streamlit` app.